# Week 1 Lab — Probabilistic Foundations for Next-Token Prediction (PyTorch)

**Goals**
- Implement a simple next-token classifier (multinomial logistic regression) on a toy sequence task.
- Inspect predicted probability distributions.
- Measure **accuracy**, **entropy**, and **calibration** (ECE).

**Deliverables**
- This notebook with results filled in.
- A 1-page reflection: *“Where probability misleads intuition”* (put it in the final cell).


## 0. Setup

This notebook is intentionally minimal and CPU-friendly. It uses a *toy* dataset so you can focus on interpretation rather than scale.


In [ ]:
import math
import random
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 1. Toy data: next-token prediction with a small vocabulary

We’ll build sequences from a small vocabulary and predict the next token from a fixed-length context window.

Two suggested generators (pick one):
1) **Bigram-ish language** with structure (easy to learn)
2) **Noisy grammar** (still learnable, but imperfect)

You can later swap in a real text corpus, but keep the vocabulary small at first.


In [ ]:
@dataclass
class ToyConfig:
    vocab: List[str]
    ctx_len: int = 8
    seq_len: int = 64
    n_sequences: int = 2000
    p_noise: float = 0.05

CFG = ToyConfig(
    vocab=list("abcdefg"),
    ctx_len=8,
    seq_len=64,
    n_sequences=3000,
    p_noise=0.08,
)

stoi = {ch:i for i,ch in enumerate(CFG.vocab)}
itos = {i:ch for ch,i in stoi.items()}
V = len(CFG.vocab)

def generate_sequence(cfg: ToyConfig) -> List[int]:
    """A simple structured generator: cycles a->b->c->... with occasional noise."""
    seq = []
    cur = random.randrange(len(cfg.vocab))
    for _ in range(cfg.seq_len):
        if random.random() < cfg.p_noise:
            cur = random.randrange(len(cfg.vocab))
        else:
            cur = (cur + 1) % len(cfg.vocab)
        seq.append(cur)
    return seq

sequences = [generate_sequence(CFG) for _ in range(CFG.n_sequences)]
sequences[0][:20], ''.join(itos[i] for i in sequences[0][:20])

## 2. Dataset: context → next token

We transform each sequence into training examples:
- input: tokens `[t-ctx_len, …, t-1]`
- target: token `t`

This mirrors the *next-token prediction* objective used in LLM pretraining (but tiny).


In [ ]:
class NextTokenDataset(Dataset):
    def __init__(self, seqs: List[List[int]], ctx_len: int):
        self.ctx_len = ctx_len
        self.examples: List[Tuple[List[int], int]] = []
        for s in seqs:
            for t in range(ctx_len, len(s)):
                x = s[t-ctx_len:t]
                y = s[t]
                self.examples.append((x, y))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx: int):
        x, y = self.examples[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

# Split sequences into train/val
n_train = int(0.9 * len(sequences))
train_seqs = sequences[:n_train]
val_seqs = sequences[n_train:]

train_ds = NextTokenDataset(train_seqs, CFG.ctx_len)
val_ds = NextTokenDataset(val_seqs, CFG.ctx_len)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=0)

len(train_ds), len(val_ds)

## 3. Model: multinomial logistic regression (bag-of-words context)

To keep Week 1 focused, we use a very simple model:
- Embed tokens
- Pool embeddings across the context window (mean)
- Linear layer → logits over vocabulary

This is not a Transformer; it’s a clean bridge from logistic regression to language modeling.


In [ ]:
class MeanContextLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T]
        h = self.emb(x)               # [B, T, D]
        h = h.mean(dim=1)             # [B, D]
        logits = self.proj(h)         # [B, V]
        return logits

model = MeanContextLM(V, d_model=48).to(device)
sum(p.numel() for p in model.parameters())

## 4. Metrics: accuracy, entropy, ECE (Expected Calibration Error)

We’ll compute:
- **Accuracy**: fraction of correct top-1 predictions
- **Entropy**: $H(p) = -\sum_i p_i \log p_i$ (distribution sharpness)
- **ECE**: bin predictions by confidence and compare mean confidence to empirical accuracy

ECE is a practical sanity check for the gap between “sounds confident” and “is correct.”


In [ ]:
def batch_entropy(probs: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    # probs: [B, V]
    p = probs.clamp_min(eps)
    return -(p * p.log()).sum(dim=-1)  # [B]

def expected_calibration_error(probs: torch.Tensor, targets: torch.Tensor, n_bins: int = 15) -> float:
    """ECE for multiclass via max-prob confidence."""
    conf, pred = probs.max(dim=1)  # [B]
    correct = (pred == targets).float()

    bins = torch.linspace(0, 1, n_bins + 1, device=probs.device)
    ece = torch.zeros((), device=probs.device)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.any():
            acc_bin = correct[mask].mean()
            conf_bin = conf[mask].mean()
            ece += (mask.float().mean()) * (conf_bin - acc_bin).abs()
    return float(ece.detach().cpu())

def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    total = 0
    correct = 0
    entropies = []
    all_probs = []
    all_targets = []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=-1)
            pred = probs.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += y.numel()
            entropies.append(batch_entropy(probs).detach().cpu())
            all_probs.append(probs.detach().cpu())
            all_targets.append(y.detach().cpu())
    ent = torch.cat(entropies).mean().item()
    probs_cat = torch.cat(all_probs)
    targets_cat = torch.cat(all_targets)
    ece = expected_calibration_error(probs_cat, targets_cat)
    return {
        'accuracy': correct / total,
        'mean_entropy_nats': ent,
        'ece': ece,
        'n_samples': total
    }


## 5. Training loop

We train with cross-entropy (negative log-likelihood), which is exactly the objective described in Week 1.


In [ ]:
def train(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, epochs: int = 5, lr: float = 3e-3):
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            running_loss += loss.item() * y.size(0)

        train_metrics = evaluate(model, train_loader)
        val_metrics = evaluate(model, val_loader)
        avg_loss = running_loss / train_metrics['n_samples']

        print(f"Epoch {epoch:02d} | loss {avg_loss:.4f} | train acc {train_metrics['accuracy']:.3f} | val acc {val_metrics['accuracy']:.3f} | val entropy {val_metrics['mean_entropy_nats']:.3f} | val ECE {val_metrics['ece']:.3f}")

train(model, train_loader, val_loader, epochs=6, lr=3e-3)
evaluate(model, val_loader)

## 6. Inspect probability distributions

Pick a few contexts and look at:
- Top-5 predicted next tokens
- Their probabilities
- Entropy

Then compare to the true generator rule (cycle + noise).


In [ ]:
def pretty_topk(probs: torch.Tensor, k: int = 5):
    vals, idx = torch.topk(probs, k)
    return [(itos[int(i)], float(v)) for v, i in zip(vals, idx)]

model.eval()
x, y = val_ds[random.randrange(len(val_ds))]
x = x.unsqueeze(0).to(device)
y = int(y)
with torch.no_grad():
    probs = F.softmax(model(x), dim=-1).squeeze(0).cpu()

ctx_str = ''.join(itos[int(i)] for i in x.squeeze(0).cpu().tolist())
print('context:', ctx_str)
print('true next:', itos[y])
print('entropy (nats):', float(batch_entropy(probs.unsqueeze(0))[0]))
print('top-5:', pretty_topk(probs, k=5))

## 7. (Optional) Temperature scaling and sampling

Observe how temperature changes output diversity without adding knowledge.


In [ ]:
def sample_next(logits: torch.Tensor, temperature: float = 1.0) -> int:
    logits = logits / max(temperature, 1e-8)
    probs = F.softmax(logits, dim=-1)
    return int(torch.multinomial(probs, num_samples=1).item())

def roll_out(model: nn.Module, start: List[int], steps: int = 40, temperature: float = 1.0) -> List[int]:
    model.eval()
    seq = start[:]
    for _ in range(steps):
        ctx = seq[-CFG.ctx_len:]
        x = torch.tensor(ctx, dtype=torch.long, device=device).unsqueeze(0)
        with torch.no_grad():
            logits = model(x).squeeze(0)
        nxt = sample_next(logits, temperature=temperature)
        seq.append(nxt)
    return seq

start = sequences[-1][:CFG.ctx_len]
for T in [0.5, 1.0, 1.5]:
    out = roll_out(model, start=start, steps=60, temperature=T)
    s = ''.join(itos[i] for i in out)
    print(f"T={T}: {s[:90]}")

## 8. Systems perspective mini-check: what did we just learn?

Write short answers:

1) Where did the model appear **overconfident**?

2) Where did high entropy correlate with error (or not)?

3) If this model were embedded in a product workflow, what would you add?
   - verification layer?
   - logging?
   - guardrails?
   - UI cues?


## 9. Reflection (1 page)

**Title:** *Where probability misleads intuition*

Suggested structure:
- One concrete example from your experiment where the model was confidently wrong (or surprisingly uncertain).
- Why this happens under the cross-entropy / MLE objective.
- What system-level control(s) would mitigate the harm.
